# Topik Kategori

In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import csv

file_kategori = '1_daftar_kategori.csv'
url_indeks_utama = "https://www.alodokter.com/komunitas/topik"
base_url = "https://www.alodokter.com/komunitas/topic-tag/"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

print("Membongkar brankas data kategori...")
response = requests.get(url_indeks_utama, headers=headers)

if response.status_code != 200:
    print(f"Gagal mengakses halaman. Status code: {response.status_code}")
else:
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # 1. Cari elemen buatan yang menyimpan JSON
    elemen_pencarian = soup.find('search-a-z-topic')
    
    if elemen_pencarian and elemen_pencarian.has_attr('search-results'):
        # 2. Ekstrak string JSON mentah
        data_json_mentah = elemen_pencarian['search-results']
        
        try:
            # 3. Ubah string JSON menjadi list dictionary Python
            daftar_topik = json.loads(data_json_mentah)
            
            # 4. Simpan ke CSV
            with open(file_kategori, mode='w', newline='', encoding='utf-8') as file:
                writer = csv.writer(file)
                writer.writerow(['Nama_Poli', 'URL_Kategori'])
                
                for topik in daftar_topik:
                    nama_poli = topik.get('post_title')
                    permalink = topik.get('permalink')
                    
                    if nama_poli and permalink:
                        url_lengkap = base_url + permalink
                        writer.writerow([nama_poli, url_lengkap])
            
            print(f"BINGGO! Berhasil mengekstrak {len(daftar_topik)} kategori topik.")
            print(f"Data telah diselamatkan ke {file_kategori}.")
            
        except json.JSONDecodeError as e:
            print(f"Gagal memecah JSON: {e}")
    else:
        print("Elemen <search-a-z-topic> tidak ditemukan. Struktur web mungkin berubah lagi.")

# Semua Pertanyaan Tiap Topik

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import random
import csv
import pandas as pd

BATAS_HALAMAN = 5 

file_kategori = '/kaggle/working/1_daftar_kategori.csv'
file_tautan = '2_daftar_semua_pertanyaan_1.csv'

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

# Baca daftar kategori dari Skrip 1
df_kategori = pd.read_csv(file_kategori)
daftar_kategori = df_kategori['URL_Kategori'].tolist()

with open(file_tautan, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow(['Kategori_Poli', 'URL_Pertanyaan'])

print("Memulai ekspedisi pencarian tautan pertanyaan...")

for url_kategori in daftar_kategori:
    nama_poli = url_kategori.split('/')[-1] 
    print(f"\nMemasuki Poli: {nama_poli}")
    
    for halaman in range(1, BATAS_HALAMAN + 1):
        # Format URL paginasi Alodokter: url_kategori/page/2
        url_target = f"{url_kategori}/page/{halaman}" if halaman > 1 else url_kategori
        print(f" -> Menyapu halaman {halaman}...")
        
        time.sleep(random.uniform(3, 6)) # JEDA ANTI-BLOKIR
        
        try:
            response = requests.get(url_target, headers=headers)
            if response.status_code != 200:
                print("    [!] Diblokir atau Halaman Habis.")
                break # Berhenti jika halaman sudah habis/diblokir
                
            soup = BeautifulSoup(response.text, 'html.parser')
            kartu_pertanyaan = soup.find_all('card-topic')
            
            if not kartu_pertanyaan:
                break # Tidak ada pertanyaan lagi di halaman ini
                
            with open(file_tautan, mode='a', newline='', encoding='utf-8') as file:
                writer = csv.writer(file)
                for kartu in kartu_pertanyaan:
                    href = kartu.get('href')
                    if href:
                        writer.writerow([nama_poli, "https://www.alodokter.com" + href])
                        
        except Exception as e:
            print(f"    [!] Error: {e}")

print(f"\nSelesai! Seluruh tautan pertanyaan tersimpan di {file_tautan}.")

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import random
import csv
import pandas as pd
import os

BATAS_HALAMAN = 5 

file_kategori = '/kaggle/input/datasets/apasijann/daftar-kategori/1_daftar_kategori.csv'
file_tautan = '2_daftar_semua_pertanyaan_2.csv'

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

# Baca daftar kategori dari Skrip 1
df_kategori = pd.read_csv(file_kategori)
daftar_kategori = df_kategori['URL_Kategori'].tolist()

# MENCEGAH OVERWRITE
# Cek apakah file sudah ada. Jika belum, buat header. Jika sudah, biarkan.
if not os.path.exists(file_tautan):
    with open(file_tautan, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(['Kategori_Poli', 'URL_Pertanyaan'])

# FAST-FORWARD
TARGET_RESUME = "simvastatin"  # Nama poli tempat terakhir berhenti
lewati_kategori = True

print(f"Mesin dihidupkan. Melewati semua antrean untuk melompat ke SETELAH: '{TARGET_RESUME}'...")

for url_kategori in daftar_kategori:
    nama_poli = url_kategori.split('/')[-1].strip()
    
    # 1. MEKANISME FAST-FORWARD
    if lewati_kategori:
        if nama_poli == TARGET_RESUME:
            lewati_kategori = False
            print(f"\n[!] Titik henti '{nama_poli}' ditemukan.")
            print(" -> Melompatinya dan bersiap pada poli berikutnya...")
        # Selama lewati_kategori True, ATAU tepat saat menemukan target, loop-nya dilewati
        continue 

    # 2. PROSES SCRAPING NORMAL UNTUK POLI SETELAHNYA
    print(f"\nMemasuki Poli: {nama_poli}")
    
    for halaman in range(1, BATAS_HALAMAN + 1):
        # Format URL paginasi Alodokter: url_kategori/page/2
        url_target = f"{url_kategori}/page/{halaman}" if halaman > 1 else url_kategori
        print(f" -> Menyapu halaman {halaman}...")
        
        time.sleep(random.uniform(3, 6)) # JEDA ANTI-BLOKIR
        
        try:
            response = requests.get(url_target, headers=headers)
            if response.status_code != 200:
                print(f"    [!] Berhenti (Status: {response.status_code}). Pindah ke poli berikutnya.")
                break # Berhenti jika halaman sudah habis/diblokir
                
            soup = BeautifulSoup(response.text, 'html.parser')
            kartu_pertanyaan = soup.find_all('card-topic')
            
            if not kartu_pertanyaan:
                break # Tidak ada pertanyaan lagi di halaman ini
                
            with open(file_tautan, mode='a', newline='', encoding='utf-8') as file:
                writer = csv.writer(file)
                for kartu in kartu_pertanyaan:
                    href = kartu.get('href')
                    if href:
                        writer.writerow([nama_poli, "https://www.alodokter.com" + href])
                        
        except Exception as e:
            print(f"    [!] Error: {e}")

print(f"\nSelesai! Seluruh tautan lanjutan tersimpan aman di {file_tautan}.")

In [ ]:
import pandas as pd

# 1. Tentukan nama file yang ingin digabung
file_1 = '/kaggle/input/datasets/apasijann/daftar-link-keluhan/2_daftar_semua_pertanyaan_1.csv'
file_2 = '/kaggle/input/datasets/apasijann/daftar-link-keluhan/2_daftar_semua_pertanyaan_2.csv'
file_gabungan = '2_daftar_semua_pertanyaan_gabungan.csv'

print("Membaca file...")
# 2. Muat data ke dalam Dataframe
df1 = pd.read_csv(file_1)
df2 = pd.read_csv(file_2)

print(f"Jumlah data File 1: {len(df1)}")
print(f"Jumlah data File 2: {len(df2)}")

# 3. Gabungkan Dataframe
df_gabungan = pd.concat([df1, df2], ignore_index=True)

# 4. Hapus duplikat be
jumlah_sebelum = len(df_gabungan)
df_gabungan = df_gabungan.drop_duplicates(subset=['URL_Pertanyaan'])
jumlah_setelah = len(df_gabungan)

print(f"Menghapus duplikat: Ditemukan {jumlah_sebelum - jumlah_setelah} data ganda.")
print(f"Total data bersih siap diekstrak: {jumlah_setelah}")

# 5. Simpan menjadi file baru
df_gabungan.to_csv(file_gabungan, index=False)
print(f"\nBerhasil! File gabungan telah disimpan sebagai: {file_gabungan}")

# Jawaban Keluhan

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import random
import csv
import pandas as pd
import re
import os

# KONFIGURASI FILE MULTI-RESUME
file_tautan_mentah = '/kaggle/input/datasets/apasijann/daftar-link-keluhan/2_daftar_semua_pertanyaan_gabungan.csv'

DAFTAR_FILE_HISTORIS = [
    '/kaggle/input/datasets/apasijann/daftar-pertanyaan-and-jawaban/3_dataset_nlp_medis_final.csv',
    '/kaggle/input/datasets/apasijann/daftar-pertanyaan-and-jawaban/3_dataset_nlp_medis_final (2).csv',
    '/kaggle/input/datasets/apasijann/daftar-pertanyaan-and-jawaban/3_dataset_nlp_medis_final (3).csv',  # <--- File yang baru saja terhenti masuk ke sini
    '/kaggle/input/datasets/apasijann/daftar-pertanyaan-and-jawaban/3_dataset_nlp_medis_final (4).csv',
    '/kaggle/input/datasets/apasijann/daftar-pertanyaan-and-jawaban/3_dataset_nlp_medis_final (5).csv',
    '/kaggle/input/datasets/apasijann/daftar-pertanyaan-and-jawaban/3_dataset_nlp_medis_final (6).csv'
] 

file_output_baru = '3_dataset_nlp_medis_final (7).csv' 

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

print("Memeriksa status data di semua brankas historis...")

# 1. Baca daftar semua tautan mentah
df_tautan = pd.read_csv(file_tautan_mentah)
df_tautan = df_tautan.drop_duplicates(subset=['URL_Pertanyaan']) 

# 2. Kumpulkan URL yang sudah berhasil diekstrak dari SEMUA file historis
url_sudah_selesai = set()

# Loop untuk membaca riwayat masa lalu
for file_lama in DAFTAR_FILE_HISTORIS:
    if os.path.exists(file_lama):
        try:
            df_lama = pd.read_csv(file_lama)
            if 'URL' in df_lama.columns:
                url_sudah_selesai.update(df_lama['URL'].dropna().tolist())
                print(f"[+] Memuat {len(df_lama)} data selesai dari {file_lama}.")
        except Exception as e:
            print(f"Gagal membaca {file_lama}: {e}")
    else:
        print(f"[-] Peringatan: {file_lama} tidak ditemukan di direktori.")

# Cek URL dari file output baru (berjaga-jaga jika skrip ini terputus di tengah jalan dan kamu jalankan ulang)
if os.path.exists(file_output_baru):
    try:
        df_baru = pd.read_csv(file_output_baru)
        if 'URL' in df_baru.columns:
            url_sudah_selesai.update(df_baru['URL'].dropna().tolist())
            print(f"[+] Memuat {len(df_baru)} data progress lanjutan dari {file_output_baru}.")
    except Exception as e:
        pass

# 3. Saring antrean: Buang URL yang sudah ada di dalam set historis
df_sisa = df_tautan[~df_tautan['URL_Pertanyaan'].isin(url_sudah_selesai)].reset_index(drop=True)

print("=" * 50)
print(f"[!] Total Antrean Mentah Awal : {len(df_tautan)}")
print(f"[!] Total URL Terselesaikan   : {len(url_sudah_selesai)}")
print(f"[!] Sisa Target yang Dikejar  : {len(df_sisa)}")
print("=" * 50)

if len(df_sisa) == 0:
    print("\nSemua data sudah terekstrak. Tidak ada lagi yang perlu disedot.")
    exit()

if not os.path.exists(file_output_baru):
    with open(file_output_baru, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(['Poli_Tujuan', 'URL', 'Keluhan', 'Jawaban_Dokter'])

# ==========================================
# MESIN EKSTRAKSI UTAMA
# ==========================================
print("\nMenyalakan mesin penyedot data...")

for index, baris in df_sisa.iterrows():
    kategori = baris['Kategori_Poli']
    url_detail = baris['URL_Pertanyaan']
    
    print(f"[{index+1}/{len(df_sisa)}] Menyedot: {url_detail}")
    time.sleep(random.uniform(3, 5)) 
    
    try:
        response = requests.get(url_detail, headers=headers)
        if response.status_code != 200:
            print(f"    -> [!] Diblokir (Code {response.status_code}). Jeda 15 detik...")
            time.sleep(15)
            continue

        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Ekstraksi Keluhan
        el_keluhan = soup.find('detail-topic')
        teks_keluhan = el_keluhan.get('member-topic-content') if el_keluhan else ""
        
        # Ekstraksi Jawaban
        el_jawaban = soup.find('doctor-topic')
        teks_jawaban = el_jawaban.get('doctor-topic-content') if el_jawaban else ""

        # Pembersihan Teks
        keluhan_bersih = re.sub(r'<[^>]+>', ' ', str(teks_keluhan).replace('\\n', ' ').replace('\\u003c', '<').replace('\\u003e', '>'))
        keluhan_bersih = re.sub(r'\s+', ' ', keluhan_bersih).strip(' "')
        
        jawaban_bersih = re.sub(r'<[^>]+>', ' ', str(teks_jawaban).replace('\\n', ' ').replace('\\u003c', '<').replace('\\u003e', '>'))
        jawaban_bersih = re.sub(r'\s+', ' ', jawaban_bersih).strip(' "')

        if keluhan_bersih and jawaban_bersih:
            # PENTING: Simpan ke file output baru (3)
            with open(file_output_baru, mode='a', newline='', encoding='utf-8') as file:
                writer = csv.writer(file)
                writer.writerow([kategori, url_detail, keluhan_bersih, jawaban_bersih])
                
    except Exception as e:
        print(f"    -> Error pada koneksi/parsing: {e}")

print("\nEksekusi terhenti (Timeout Kaggle atau semua selesai).")

In [ ]:
import pandas as pd
import os

# ==========================================
# DAFTAR FILE YANG AKAN DIGABUNGKAN
# ==========================================
daftar_file_pecahan = [
    '/kaggle/input/datasets/apasijann/daftar-pertanyaan-and-jawaban/3_dataset_nlp_medis_final.csv',
    '/kaggle/input/datasets/apasijann/daftar-pertanyaan-and-jawaban/3_dataset_nlp_medis_final (2).csv',
    '/kaggle/input/datasets/apasijann/daftar-pertanyaan-and-jawaban/3_dataset_nlp_medis_final (3).csv',
    '/kaggle/input/datasets/apasijann/daftar-pertanyaan-and-jawaban/3_dataset_nlp_medis_final (4).csv',
    '/kaggle/input/datasets/apasijann/daftar-pertanyaan-and-jawaban/3_dataset_nlp_medis_final (5).csv',
    '/kaggle/input/datasets/apasijann/daftar-pertanyaan-and-jawaban/3_dataset_nlp_medis_final (6).csv',
    '/kaggle/input/datasets/apasijann/daftar-pertanyaan-and-jawaban/3_dataset_nlp_medis_final (7).csv'# Lokasi file terbaru hasil scraping sesimu
]

file_master_final = '3_dataset_nlp_medis_complete.csv'
list_dataframe = []

print("Memulai pembacaan brankas data...")
print("-" * 60)

# 1. Iterasi dan muat semua file yang ada
for jalur_file in daftar_file_pecahan:
    if os.path.exists(jalur_file):
        try:
            df = pd.read_csv(jalur_file)
            print(f"[+] Berhasil memuat: {jalur_file} | Jumlah: {len(df)} baris")
            list_dataframe.append(df)
        except Exception as e:
            print(f"[!] Gagal membaca {jalur_file}: {e}")
    else:
        print(f"[-] Peringatan: File {jalur_file} tidak ditemukan (Dilewati).")

# 2. Proses Penggabungan Utama
if list_dataframe:
    print("-" * 60)
    print("Menggabungkan seluruh matriks data...")
    df_master = pd.concat(list_dataframe, ignore_index=True)
    total_awal = len(df_master)
    print(f"[✓] Total baris kotor berhasil disatukan: {total_awal}")

    # 3. Deduplikasi
    df_master = df_master.drop_duplicates(subset=['URL'])
    total_akhir = len(df_master)
    
    print(f"[✓] Pembersihan selesai: Menghapus {total_awal - total_akhir} data baris ganda.")
    print(f"[✓] Ukuran Dataset Master Bersih: {total_akhir} baris data.")

    # 4. Simpan ke File Master Utama
    df_master.to_csv(file_master_final, index=False)
    print("-" * 60)
    print(f"[SUKSES] Dataset Master Final disimpan sebagai: {file_master_final}")
    print("Dataset ini siap 100% untuk masuk ke tahap Prapemrosesan Teks NLP.")
else:
    print("[Gagal] Tidak ada data yang berhasil dimuat. Periksa kembali lokasi filemu.")